In [3]:
import pandas as pd
import numpy as np

from gensim.models import Word2Vec

In [4]:
corpus = pd.read_csv("../data/output/CORPUS.csv", sep="|")

corpus.head()

,book_number,chapter,verse,token_id,token_str,term_str,pos,pos_group
0,1,1,1,1,in,in,IN,OTHER
1,1,1,1,2,the,the,DT,OTHER
2,1,1,1,3,beginning,beginning,VBG,VERB
3,1,1,1,4,god,god,NN,NOUN
4,1,1,1,5,created,created,VBN,VERB


In [5]:
sentences = (
    corpus
    .groupby(["book_number", "chapter", "verse"])["term_str"]
    .apply(list)
    .tolist()
)

In [6]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1
)

In [7]:
vocab = pd.read_csv("../data/output/VOCAB.csv", sep="|")
vocab.head()

,term_str,n,p,i,df,dfidf,max_pos,max_pos_group,stop
0,a,8177,0.010355,6.593531,6216,40985.387317,DT,OTHER,True
1,aaron,319,0.000404,11.273474,303,3415.862649,NN,NOUN,False
2,aaron's,31,0.000039,14.636690,31,453.737402,NN,NOUN,False
3,aaronites,2,0.000003,18.590887,2,37.181773,NNS,NOUN,False
4,abaddon,1,0.000001,19.590887,1,19.590887,NN,NOUN,False


In [8]:
vocab_terms = [
    term for term in vocab["term_str"]
    if term in w2v_model.wv
]

In [9]:
vectors = [
    w2v_model.wv[term]
    for term in vocab_terms
]

vocab_w2v = pd.DataFrame(
    vectors,
    index=vocab_terms
)

vocab_w2v.index.name = "term_str"
vocab_w2v.reset_index(inplace=True)

vocab_w2v.head()

,term_str,0,1,2,3,4,5,6,7,8,...,90,91,92,93,94,95,96,97,98,99
0,a,-0.076918,0.307668,-0.018858,-0.363742,0.394469,0.300149,0.382173,0.624130,-0.057221,...,-0.008510,0.173102,-0.064713,-0.450167,0.352525,0.344765,0.006750,-0.327284,-0.300511,0.336414
1,aaron,-0.121549,0.237823,0.579024,0.127727,-0.039846,-0.351108,0.078991,0.310827,-0.640818,...,-0.348973,0.087374,-0.453949,-0.355877,0.243234,0.494017,-0.325608,0.308654,-0.599605,-0.067395
2,aaron's,-0.086982,0.074472,0.413678,0.319847,0.112313,-0.028059,-0.041915,0.359314,-0.254140,...,-0.241337,-0.176876,-0.374701,-0.265356,0.374996,0.332725,0.027086,-0.097409,-0.107350,-0.131387
3,abated,0.037336,0.253331,-0.196200,0.042158,0.162734,-0.050281,-0.141764,0.339077,-0.233735,...,-0.035835,0.165954,0.063998,-0.098712,0.295326,0.227306,0.121758,-0.231402,-0.003298,-0.064221
4,abdon,0.036784,0.111374,0.071258,0.107317,0.155672,-0.201936,-0.259829,0.270885,-0.162629,...,-0.033058,0.263291,0.001156,-0.265447,0.216999,0.294079,-0.024512,-0.064198,0.054212,-0.021314


In [12]:
print("Vector size:", w2v_model.vector_size)
print("Window size:", w2v_model.window)
print("Min count:", w2v_model.min_count)
print("Architecture:", "Skip-gram" if w2v_model.sg == 1 else "CBOW")

Vector size: 100
Window size: 5
Min count: 5
Architecture: Skip-gram


In [13]:
w2v_model.wv.most_similar("god", topn=10)

[('lord', 0.714459240436554),
 ('redeemer', 0.7104629278182983),
 ('saviour', 0.7061470150947571),
 ('gracious', 0.6912044882774353),
 ('excellency', 0.6865217685699463),
 ('magnified', 0.6715919375419617),
 ('merciful', 0.6653755903244019),
 ('zeal', 0.6630026698112488),
 ('lovingkindness', 0.6607007384300232),
 ('preserved', 0.6587172150611877)]

In [14]:
w2v_model.wv.most_similar("love", topn=10)

[('patience', 0.7593409419059753),
 ('comfort', 0.758465051651001),
 ('uprightness', 0.7504633069038391),
 ('despise', 0.7369858026504517),
 ('brotherly', 0.733249306678772),
 ('meekness', 0.7317979335784912),
 ('precepts', 0.728934109210968),
 ('reprove', 0.7284730672836304),
 ('kindness', 0.7260928153991699),
 ('herein', 0.7235601544380188)]

In [15]:
print("Original VOCAB size:", len(vocab))
print("Terms with embeddings:", len(vocab_w2v))

Original VOCAB size: 12878
Terms with embeddings: 5347


In [11]:
delimiter = "|"
ohco_bag = "book_number → chapter → verse"
num_features = 100
library_used = "gensim.models.Word2Vec (Skip-gram)"

print("Delimiter:", f'"{delimiter}"')
print("Document bag expressed in OHCO levels:", ohco_bag)
print("Number of features generated:", num_features)
print("The library used to generate the embeddings:", library_used)

Delimiter: "|"
Document bag expressed in OHCO levels: book_number → chapter → verse
Number of features generated: 100
The library used to generate the embeddings: gensim.models.Word2Vec (Skip-gram)


In [10]:
output_path = "../data/output/VOCAB_W2V.csv"

vocab_w2v.to_csv(
    output_path,
    sep="|",
    index=False
)

print("Saved:", output_path)
print("Delimiter used:", '"|"')
print("Number of features generated:", 100)
print("Library used: gensim Word2Vec")

Saved: ../data/output/VOCAB_W2V.csv
Delimiter used: "|"
Number of features generated: 100
Library used: gensim Word2Vec
